In [1]:
import pandas as pd
import numpy as np
import re

def answer_one():
    energy = pd.read_excel('Energy Indicators.xls', skiprows=17, skipfooter=38, encoding='Windows-1254')
    energy = energy.drop(energy.columns[[0, 1]], axis=1)
    energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable']
    energy.replace('...', np.nan, inplace=True)
    energy['Energy Supply'] = energy['Energy Supply'] * 1000000  # Переведення в гігаджоули
    pd.options.display.float_format = '{:,.0f}'.format

    energy['Country'] = energy['Country'].apply(lambda x: re.sub(r'\(.*\)', '', x))
    energy['Country'] = energy['Country'].apply(lambda x: re.sub(r'\d+', '', x))

    country_rename_map = {
        'Republic of Korea': 'South Korea',
        'United States of America': 'United States',
        'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
        'China, Hong Kong Special Administrative Region': 'Hong Kong'
    }
    energy['Country'] = energy['Country'].replace(country_rename_map)
    
    GDP = pd.read_csv('world_bank.csv', skiprows=4)
    GDP['Country Name'] = GDP['Country Name'].replace({
        'Korea, Rep.': 'South Korea',
        'Iran, Islamic Rep.': 'Iran',
        'Hong Kong SAR, China': 'Hong Kong'
    })
    
    ScimEn = pd.read_excel('scimagojr country rank 1996-2023.xlsx')
    ScimEn = ScimEn[ScimEn['Rank'] <= 15]

    GDP.rename(columns={'Country Name': 'Country'}, inplace=True)

    df = pd.merge(ScimEn, energy, on='Country', how='inner')
    df = pd.merge(df, GDP, on='Country', how='inner')
    df.set_index('Country', inplace=True)

    df = df[['Rank', 'Documents', 'Citable documents', 'Citations', 'Self-citations', 
             'Citations per document', 'H index', 'Energy Supply', 'Energy Supply per Capita', 
             '% Renewable', '2006', '2007', '2008', '2009', '2010', '2011', 
             '2012', '2013', '2014', '2015']]
    
    return df

answer_one()


,Rank,Documents,Citable documents,Citations,Self-citations,Citations per document,H index,Energy Supply,Energy Supply per Capita,% Renewable,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
Country,,,,,,,,,,,,,,,,,,,,
China,1,402188,400158,5077232,3511673,13,343,"127,191,000,000",93,20,"2,752,118,657,184","3,550,327,802,998","4,594,336,785,738","5,101,691,124,285","6,087,191,746,679","7,551,545,703,441","8,532,185,381,681","9,570,471,111,832","10,475,624,944,355","11,061,572,618,579"
United States,2,208042,203356,3540288,1004877,17,458,"90,838,000,000",286,12,"13,815,583,000,000","14,474,228,000,000","14,769,862,000,000","14,478,067,000,000","15,048,971,000,000","15,599,732,000,000","16,253,970,000,000","16,880,683,000,000","17,608,138,000,000","18,295,019,000,000"
India,3,81853,79757,968523,369256,12,237,"33,195,000,000",26,15,"940,259,888,788","1,216,736,438,835","1,198,895,139,006","1,341,888,016,995","1,675,615,519,485","1,823,051,829,895","1,827,637,590,410","1,856,721,507,622","2,039,126,479,155","2,103,588,360,045"
Japan,4,58342,57735,717946,154848,12,228,"18,984,000,000",149,10,"4,601,663,122,650","4,579,750,920,355","5,106,679,115,127","5,289,493,117,994","5,759,071,769,013","6,233,147,172,341","6,272,362,996,105","5,212,328,181,166","4,896,994,405,353","4,444,930,651,964"
United Kingdom,5,56288,54713,1095371,179764,19,293,"7,920,000,000",124,11,"2,708,441,582,337","3,090,510,204,082","2,929,411,764,706","2,412,840,006,232","2,485,482,596,185","2,663,805,834,828","2,707,089,726,615","2,784,853,502,534","3,064,708,247,921","2,927,911,140,917"
Germany,6,50906,49773,777362,160302,15,252,"13,261,000,000",165,18,"3,046,308,753,671","3,484,056,680,855","3,808,786,022,312","3,479,800,820,863","3,468,154,343,000","3,824,828,563,521","3,597,896,500,945","3,808,086,291,482","3,965,800,686,310","3,423,568,450,957"
Russian Federation,7,46186,45868,217996,96688,5,112,"30,709,000,000",214,17,"989,932,071,353","1,299,703,478,482","1,660,848,058,303","1,222,645,900,056","1,524,916,715,224","2,045,922,753,398","2,208,293,553,878","2,292,470,078,346","2,059,241,589,895","1,363,482,182,198"
Canada,8,41209,40390,915491,142691,22,284,"10,431,000,000",296,62,"1,319,264,809,591","1,468,820,407,783","1,552,989,690,722","1,374,625,142,157","1,617,343,367,486","1,793,326,630,175","1,828,366,481,522","1,846,597,421,835","1,805,749,878,440","1,556,508,816,217"
Italy,9,38700,36909,639473,147302,17,209,"6,530,000,000",109,34,"1,958,563,654,386","2,222,524,108,128","2,417,508,414,187","2,209,484,319,013","2,144,936,254,535","2,306,974,020,278","2,097,929,495,122","2,153,225,581,941","2,173,255,507,986","1,845,428,048,839"


In [2]:
def answer_two():
    Top15 = answer_one()

    gdp_columns = [str(year) for year in range(2006, 2016)]

    avgGDP = Top15[gdp_columns].mean(axis=1, skipna=True)

    avgGDP = avgGDP.sort_values(ascending=False)

    avgGDP = avgGDP.to_frame()
    avgGDP.columns = ['avgGDP']

    return avgGDP

answer_two()

,avgGDP
Country,
United States,"15,722,425,300,000"
China,"6,927,706,587,677"
Japan,"5,239,642,145,207"
Germany,"3,590,728,711,392"
United Kingdom,"2,777,505,460,636"
France,"2,692,000,013,150"
Italy,"2,152,982,940,441"
Brazil,"1,988,888,979,247"
Russian Federation,"1,666,745,638,113"


In [3]:
def answer_three():
    Top15 = answer_one()

    gdp_columns = [str(year) for year in range(2006, 2016)]

    avgGDP = Top15[gdp_columns].mean(axis=1, skipna=True)

    avgGDP = avgGDP.sort_values(ascending=False)

    country_6th = avgGDP.index[5] 

    gdp_2006 = Top15.loc[country_6th, '2006']
    gdp_2015 = Top15.loc[country_6th, '2015']
    gdp_change = gdp_2015 - gdp_2006

    return gdp_change

answer_three()

124621907951.68018

In [4]:
def answer_four():
    Top15 = answer_one()

    Top15['Self-Citations to Total Citations'] = Top15['Self-citations'] / Top15['Citations']

    max_ratio = Top15['Self-Citations to Total Citations'].max()

    country_max_ratio = Top15['Self-Citations to Total Citations'].idxmax()

    return country_max_ratio, max_ratio

answer_four()

('China', 0.6916510807463594)

In [5]:
def answer_five():
    Top15 = answer_one()

    Top15['Estimated Population'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']

    sorted_by_population = Top15.sort_values('Estimated Population', ascending=False)

    return sorted_by_population.index[2]

answer_five()

'United States'

In [6]:
def answer_six():
    Top15 = answer_one()

    Top15['Estimated Population'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']

    Top15['Citable Docs per Capita'] = Top15['Citable documents'] / Top15['Estimated Population']

    correlation = Top15['Citable Docs per Capita'].corr(Top15['Energy Supply per Capita'])

    return correlation

answer_six()

0.6963196993588852

In [7]:
import pandas as pd

def answer_seven():
    Top15 = answer_one()  # Отримуємо DataFrame з answer_one()
    
    # Словник відповідності країн та континентів
    ContinentDict = {'China': 'Asia', 
                     'United States': 'North America', 
                     'Japan': 'Asia', 
                     'United Kingdom': 'Europe', 
                     'Russian Federation': 'Europe', 
                     'Canada': 'North America', 
                     'Germany': 'Europe', 
                     'India': 'Asia',
                     'France': 'Europe', 
                     'South Korea': 'Asia', 
                     'Italy': 'Europe', 
                     'Spain': 'Europe', 
                     'Australia': 'Australia', 
                     'Brazil': 'South America'}

    # Додаємо колонку 'Continent' згідно зі словником
    Top15['Continent'] = Top15.index.map(ContinentDict)

    # Оцінюємо населення
    Top15['Estimated Population'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']

    # Групуємо за континентами та обчислюємо потрібні значення
    result = Top15.groupby('Continent')['Estimated Population'].agg(['size', 'sum', 'mean', 'std'])

    return result

answer_seven()

,size,sum,mean,std
Continent,,,,
Asia,4,"2,821,590,756","705,397,689","713,877,879"
Australia,1,"23,316,017","23,316,017",nan
Europe,6,"457,929,667","76,321,611","34,647,667"
North America,2,"352,855,249","176,427,625","199,669,645"
South America,1,"205,915,254","205,915,254",nan
